# Room Occupation Prediction Training

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (confusion_matrix, classification_report, 
                           roc_curve, auc, roc_auc_score, accuracy_score, mean_squared_error, mean_absolute_error, r2_score)
from sklearn.utils.class_weight import compute_class_weight
import os
import json
import glob
from datetime import datetime, timedelta
import missingno as msno
from matplotlib.dates import DateFormatter
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from math import sqrt
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pickle

## Extract and Combine Data

In [ ]:
def extract_motion_data(json_data, apartment_id):
    data = {}
    
    # Metadata
    data['DateTime'] = json_data.get('Datetime', '')
    data['Hours'] = json_data.get('Hours', '')
    data['User'] = json_data.get('User', '')
    data['ApartmentID'] = apartment_id
    
    # Extract motion data only
    if 'Motions' in json_data:
        for room, motion_data in json_data['Motions'].items():
            # if room in excluded_devices:
            #     continue
            for key, value in motion_data.items():
                data[f'{room}_Motion_{key}'] = value
    
    return data

def process_motion_data(base_dir, apartment_id, year, selected_days):
    print(f"Processing {apartment_id} motion data...")
    
    # Create paths
    apartment_year_path = os.path.join(base_dir, apartment_id, year)
    print(f"Checking path: {apartment_year_path}")
    
    if not os.path.exists(apartment_year_path):
        print(f"Path not found: {apartment_year_path}")
        return
    
    # Create output directory
    output_dir = f"motion_data_{apartment_id}"
    os.makedirs(output_dir, exist_ok=True)
    
    # Process each month and day
    for month in selected_days.keys():
        month_dir = os.path.join(apartment_year_path, month)
        print(f"Checking month directory: {month_dir}")
        
        if not os.path.exists(month_dir):
            print(f"Month directory not found: {month_dir}")
            continue
        
        for day in selected_days[month]:
            day_str = f"{day:02d}"
            day_data = []
            
            # Check if there's a day subfolder
            day_dir = os.path.join(month_dir, day_str)
            if os.path.exists(day_dir):
                # Look in the day folder
                pattern = os.path.join(day_dir, "*.json")
            else:
                # Look directly in the month folder
                pattern = os.path.join(month_dir, f"{day_str}.{month}.{year}*.json")
            
            json_files = glob.glob(pattern)
            print(f"Found {len(json_files)} files for {month}/{day_str}")
            
            # Skip if no files found
            if not json_files:
                continue
            
            # Process all JSON files for this day
            for json_file in json_files:
                try:
                    with open(json_file, 'r') as f:
                        data = json.load(f)
                    
                    # Extract motion data
                    motion_data = extract_motion_data(data, apartment_id)
                    if motion_data and any(key for key in motion_data.keys() if 'Motion' in key):
                        day_data.append(motion_data)
                except Exception as e:
                    print(f"Error processing {json_file}: {e}")
            
            # If we have data for this day, process it
            if day_data:
                df = pd.DataFrame(day_data)
                
                # Convert DateTime to datetime objects
                df['DateTime'] = pd.to_datetime(df['DateTime'], format='%d/%m/%Y', errors='coerce')
                
                # Extract hour from Hours column (format: 'HH:MM')
                if 'Hours' in df.columns:
                    df['Hour'] = df['Hours'].str.split(':').str[0].astype(int)
                else:
                    df['Hour'] = df['DateTime'].dt.hour
                
                # Add time-based features
                df['Date'] = df['DateTime'].dt.strftime('%d/%m/%Y')
                df['DayOfWeek'] = df['DateTime'].dt.dayofweek  # 0 is Monday, 6 is Sunday
                df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)  # 1 for weekends
                
                # Add a counter column for potential future use
                df['EntryCounter'] = 1
                
                # Convert boolean motion columns to numeric (1/0)
                motion_cols = [col for col in df.columns if 'Motion' in col]
                for col in motion_cols:
                    # Handle different data types appropriately
                    if df[col].dtype == 'bool' or (df[col].dtype == 'object' and set(df[col].dropna().unique()) <= {'True', 'False', True, False}):
                        # Try to convert string booleans to numeric
                        try:
                            df[col] = df[col].map({'True': 1, 'False': 0, True: 1, False: 0})
                        except:
                            pass
                
                # Save to CSV - no aggregation, preserving all original data points
                output_filename = os.path.join(output_dir, f"{apartment_id}_{year}_{month}_{day_str}_motion.csv")
                df.to_csv(output_filename, index=False)
                print(f"Saved raw motion data to {output_filename}")

def combine_room_data(apartment_id, rooms):
    input_dir = f"motion_data_{apartment_id}"
    
    for room in rooms:
        all_data = []
        
        # Get all CSV files for this apartment
        csv_files = glob.glob(os.path.join(input_dir, f"{apartment_id}_*.csv"))
        
        for csv_file in csv_files:
            try:
                df = pd.read_csv(csv_file)
                
                # Filter columns for this room
                room_cols = [col for col in df.columns if room in col and 'Motion' in col]
                
                # Only proceed if we have motion data for this room
                if room_cols:
                    # Keep metadata columns and room-specific columns
                    keep_cols = ['DateTime', 'Date', 'Hour', 'Hours', 'DayOfWeek', 'IsWeekend', 
                                'EntryCounter', 'ApartmentID', 'User'] + room_cols
                    
                    # Keep only columns that exist in the DataFrame
                    keep_cols = [col for col in keep_cols if col in df.columns]
                    
                    room_df = df[keep_cols].copy()
                    all_data.append(room_df)
            except Exception as e:
                print(f"Error processing {csv_file} for room {room}: {e}")
        
        if all_data:
            # Combine all daily data
            combined_df = pd.concat(all_data, ignore_index=True)
            
            # Save combined data for this room
            output_filename = f"{apartment_id}_{room}_motion_data.csv"
            combined_df.to_csv(output_filename, index=False)
            print(f"Combined motion data for {room} saved to {output_filename}")

# Define days to include for each month
selected_days = {
    '05': list(range(1, 32)),
    '06': list(range(1, 32)),
    '07': list(range(1, 32)),
    '08': list(range(1, 32)),
    '09': list(range(1, 32))
}

# Base directory where data is stored
base_dir = "./Data/Sensors"
year = "2023"

# Process apartments
process_motion_data(base_dir, "Apartment_1", year, selected_days)
process_motion_data(base_dir, "Apartment_2", year, selected_days)

# Combine room data after processing
combine_room_data("Apartment_1", ["Kitchen", "Office"])
combine_room_data("Apartment_2", ["Living", "Office"])

## Check for duplicates, missing values, outliers

In [ ]:
def check_duplicates(df):
    print("=== DUPLICATE CHECKS ===")
   
    # Check for exact duplicates
    exact_duplicates = df.duplicated().sum()
    print(f"Exact duplicate rows: {exact_duplicates}")
   
    # Check for time-based duplicates if DateTime and Hours columns exist
    if 'DateTime' in df.columns and 'Hours' in df.columns:
        time_duplicates = df.duplicated(subset=['DateTime', 'Hours']).sum()
        print(f"Duplicate DateTime/Hours combinations: {time_duplicates}")
       
        if time_duplicates > 0:
            print("\nFirst few duplicate time entries:")
            return df[df.duplicated(subset=['DateTime', 'Hours'], keep=False)].sort_values(['DateTime', 'Hours']).head()
   
    return None

def check_missing_values(df):
    print("\n=== MISSING VALUES CHECKS ===")
   
    # Count missing values per column
    missing_values = df.isnull().sum()
    missing_values = missing_values[missing_values > 0]
   
    if len(missing_values) > 0:
        print("Columns with missing values:")
        for col, count in missing_values.items():
            print(f"  {col}: {count} missing values ({count/len(df)*100:.2f}%)")
    else:
        print("No missing values found.")
   
    return missing_values

def check_outliers(df):
    print("\n=== OUTLIER CHECKS ===")
   
    # Only analyze numeric columns
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    outlier_info = {}
   
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
       
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
       
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_count = len(outliers)
       
        if outlier_count > 0:
            outlier_pct = outlier_count / len(df) * 100
            outlier_info[col] = {
                'count': outlier_count,
                'percentage': outlier_pct,
                'bounds': (lower_bound, upper_bound)
            }
   
    # Display outlier results
    if outlier_info:
        print(f"Found {len(outlier_info)} columns with outliers.")
        sorted_outliers = sorted(outlier_info.items(), key=lambda x: x[1]['percentage'], reverse=True)
       
        print("\nAll columns with outliers:")
        for col, stats in sorted_outliers:
            print(f"  {col}: {stats['count']} outliers ({stats['percentage']:.2f}%)")
            print(f"    Values outside: [{stats['bounds'][0]:.2f}, {stats['bounds'][1]:.2f}]")
    else:
        print("No outliers found in any column.")
        
    print("\n")
    return outlier_info

# Load data and run analysis
df = pd.read_csv('Apartment_1_Kitchen_motion_data.csv')
dup_results = check_duplicates(df)
missing_results = check_missing_values(df)
outlier_results = check_outliers(df)

df = pd.read_csv('Apartment_1_Office_motion_data.csv')
dup_results = check_duplicates(df)
missing_results = check_missing_values(df)
outlier_results = check_outliers(df)

df = pd.read_csv('Apartment_2_Living_motion_data.csv')
dup_results = check_duplicates(df)
missing_results = check_missing_values(df)
outlier_results = check_outliers(df)

df = pd.read_csv('Apartment_2_Office_motion_data.csv')
dup_results = check_duplicates(df)
missing_results = check_missing_values(df)
outlier_results = check_outliers(df)

## Fix for duplicates, missing values, outliers

In [ ]:
def remove_rows_with_missing_values(df, threshold=None, subset=None):
    
    # Make a copy of the dataframe
    df_clean = df.copy()
    
    # If subset is specified, check only those columns
    if subset is not None:
        # Ensure all specified columns exist in the dataframe
        subset = [col for col in subset if col in df.columns]
        if not subset:
            print("None of the specified columns exist in the dataframe.")
            return df_clean
    
    # Count original rows
    original_rows = len(df_clean)
    
    if threshold is not None:
        if not 0 <= threshold <= 1:
            raise ValueError("Threshold must be between 0.0 and 1.0")
        
        # Calculate missing values ratio for each row
        if subset:
            missing_ratio = df_clean[subset].isnull().mean(axis=1)
        else:
            missing_ratio = df_clean.isnull().mean(axis=1)
        
        # Remove rows with missing ratio > threshold
        rows_to_drop = missing_ratio[missing_ratio > threshold].index
        if len(rows_to_drop) > 0:
            df_clean = df_clean.drop(index=rows_to_drop)
            rows_removed = len(rows_to_drop)
            print(f"Removed {rows_removed} rows with > {threshold*100}% missing values "
                  f"({rows_removed/original_rows*100:.2f}% of data)")
        else:
            print(f"No rows with > {threshold*100}% missing values found.")
    else:
        # Remove rows with any missing values
        if subset:
            df_clean = df_clean.dropna(subset=subset)
        else:
            df_clean = df_clean.dropna()
        
        # Calculate number of rows removed
        rows_removed = original_rows - len(df_clean)
        if rows_removed > 0:
            print(f"Removed {rows_removed} rows with missing values "
                  f"({rows_removed/original_rows*100:.2f}% of data)")
        else:
            print("No rows with missing values found.")
    
    return df_clean

def remove_duplicates(df):

    # Count duplicates before removal
    dup_count = df.duplicated().sum()
    
    # If we have time-based data, check for time-based duplicates
    if 'DateTime' in df.columns and 'Hours' in df.columns:
        time_dup_count = df.duplicated(subset=['DateTime', 'Hours']).sum()
        if time_dup_count > 0:
            print(f"Found {time_dup_count} duplicate time entries. Removing...")
            return df.drop_duplicates(subset=['DateTime', 'Hours'], keep='first')
    
    # If no time-based duplicates but we have exact duplicates
    if dup_count > 0:
        print(f"Found {dup_count} exact duplicate rows. Removing...")
        return df.drop_duplicates()
    
    print("No duplicates found.")
    return df

def handle_outliers(df, method='iqr', columns=None, threshold=1.5):

    # Make a copy of the dataframe
    df_clean = df.copy()
    
    # Select columns to check for outliers
    if columns is None:
        columns = df.select_dtypes(include=['int64', 'float64']).columns
        # Exclude columns ending with _Motion
        columns = [col for col in columns if not col.endswith('_Motion')]
    else:
        # Ensure all specified columns exist and are numeric
        columns = [col for col in columns if col in df.columns and
                   pd.api.types.is_numeric_dtype(df[col]) and
                   not col.endswith('_Motion')]
    
    outlier_info = {}
    
    # Process each column
    for col in columns:
        # Skip columns with all identical values
        if df[col].nunique() <= 1:
            continue
        
        # Calculate boundaries for outliers based on selected method
        if method == 'iqr':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            
            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR
            
            # Identify outliers
            outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
            outlier_count = len(outliers)
            
            if outlier_count > 0:
                outlier_percentage = outlier_count / len(df) * 100
                outlier_info[col] = {
                    'count': outlier_count,
                    'percentage': outlier_percentage,
                    'bounds': (lower_bound, upper_bound)
                }
                
                # Handle outliers based on specified method
                if method == 'iqr':
                    # Cap values at the boundaries
                    df_clean.loc[df_clean[col] < lower_bound, col] = lower_bound
                    df_clean.loc[df_clean[col] > upper_bound, col] = upper_bound
        
        elif method == 'zscore':
            # Calculate z-scores
            mean = df[col].mean()
            std = df[col].std()
            z_scores = (df[col] - mean) / std
            
            # Identify outliers (|z| > threshold)
            outliers = df[abs(z_scores) > threshold]
            outlier_count = len(outliers)
            
            if outlier_count > 0:
                outlier_percentage = outlier_count / len(df) * 100
                lower_bound = mean - threshold * std
                upper_bound = mean + threshold * std
                
                outlier_info[col] = {
                    'count': outlier_count,
                    'percentage': outlier_percentage,
                    'bounds': (lower_bound, upper_bound)
                }
                
                # Cap values at z-score boundaries
                df_clean.loc[df_clean[col] < lower_bound, col] = lower_bound
                df_clean.loc[df_clean[col] > upper_bound, col] = upper_bound
        
        elif method == 'remove':
            # For this method, we'll collect all outliers first, then remove rows at the end
            if col not in outlier_info:
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                
                lower_bound = Q1 - threshold * IQR
                upper_bound = Q3 + threshold * IQR
                
                # Identify outliers
                outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
                outliers = df[outlier_mask]
                outlier_count = len(outliers)
                
                if outlier_count > 0:
                    outlier_percentage = outlier_count / len(df) * 100
                    outlier_info[col] = {
                        'count': outlier_count,
                        'percentage': outlier_percentage,
                        'bounds': (lower_bound, upper_bound),
                        'mask': outlier_mask
                    }
    
    # If we're removing outlier rows, we need to do it once for all columns
    if method == 'remove' and outlier_info:
        # Combine all outlier masks
        combined_mask = pd.Series(False, index=df.index)
        for col, info in outlier_info.items():
            combined_mask = combined_mask | info['mask']
        
        # Remove rows with outliers
        df_clean = df_clean[~combined_mask]
        rows_removed = sum(combined_mask)
        print(f"Removed {rows_removed} rows containing outliers ({rows_removed/len(df)*100:.2f}% of data)")
    
    # Report on outliers found and handled
    if outlier_info:
        print(f"Found and handled outliers in {len(outlier_info)} columns using {method} method:")
        
        # Sort by percentage to show most affected columns first
        sorted_outliers = sorted(outlier_info.items(), 
                                key=lambda x: x[1]['percentage'] if 'percentage' in x[1] else 0, 
                                reverse=True)
        
        for col, info in sorted_outliers[:10]:  # Show top 10
            if method != 'remove':
                print(f"  - {col}: {info['count']} outliers ({info['percentage']:.2f}%) - capped at [{info['bounds'][0]:.2f}, {info['bounds'][1]:.2f}]")
    else:
        print("No outliers found or handled.")
    
    if len(outlier_info) > 10:
        print(f"  ... and {len(outlier_info) - 10} more columns")
    
    return df_clean, outlier_info

def clean_data(filepath, outlier_method='iqr', outlier_threshold=1.5):

    print(f"Cleaning data from: {filepath}")
    
    # Load the data
    df = pd.read_csv(filepath)
    print(f"Initial shape: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # 1. Remove rows missing values
    df_no_missing = remove_rows_with_missing_values(df)
    print(f"After removing rows with missing values: {df_no_missing.shape[0]} rows, {df_no_missing.shape[1]} columns")
    
    # 2. Remove duplicates
    df_no_duplicates = remove_duplicates(df_no_missing)
    print(f"After removing duplicates: {df_no_duplicates.shape[0]} rows, {df_no_duplicates.shape[1]} columns")
    
    # 3. Handle outliers
    df_clean, outlier_info = handle_outliers(
        df_no_duplicates, 
        method=outlier_method,
        threshold=outlier_threshold
    )
    print(f"Final shape after cleaning: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
    
    return df_clean


df_clean = clean_data('Apartment_1_Kitchen_motion_data.csv')
df_clean.to_csv('Apartment_1_Kitchen_motion_data.csv', index=False)

df_clean = clean_data('Apartment_1_Office_motion_data.csv')
df_clean.to_csv('Apartment_1_Office_motion_data.csv', index=False)

df_clean = clean_data('Apartment_2_Living_motion_data.csv')
df_clean.to_csv('Apartment_2_Living_motion_data.csv', index=False)

df_clean = clean_data('Apartment_2_Office_motion_data.csv')
df_clean.to_csv('Apartment_2_Office_motion_data.csv', index=False)

## Histograms

In [ ]:
# Set larger figure size for better visualization
plt.rcParams['figure.figsize'] = (12, 6)

# Process each file
files = [
    'Apartment_1_Kitchen_motion_data.csv',
    'Apartment_1_Office_motion_data.csv', 
    'Apartment_2_Living_motion_data.csv',
    'Apartment_2_Office_motion_data.csv'
]

for file in files:
    try:
        # Load the data
        df = pd.read_csv(file)
        room_name = file.split('_')[3]  # Extract room name for plots
        
        print(f"\n=== Processing {room_name} motion data ===")
        print(f"Columns found: {df.columns.tolist()}")
        
        # For Motion column
        motion_col = [col for col in df.columns if 'motion' in col.lower()]
        if motion_col:
            motion_col = motion_col[0]
            plt.figure()
            sns.countplot(x=df[motion_col])
            plt.title(f'Motion Detection Distribution (0/1) - {room_name}')
            plt.xlabel('Motion Detected')
            plt.xticks([0, 1], ['No Motion (0)', 'Motion Detected (1)'])
            plt.ylabel('Count')
            plt.tight_layout()
            plt.show()
        else:
            print(f"No motion column found in {file}")
        
        # For Light column
        light_col = [col for col in df.columns if 'light' in col.lower()]
        if light_col:
            light_col = light_col[0]
            plt.figure()
            sns.histplot(df[light_col], kde=True)
            plt.title(f'Light Level Distribution - {room_name}')
            plt.xlabel('Light Level')
            plt.ylabel('Frequency')
            plt.tight_layout()
            plt.show()
        else:
            print(f"No light column found in {file}")
        
        # For Temperature column
        temp_col = [col for col in df.columns if 'temp' in col.lower()]
        if temp_col:
            temp_col = temp_col[0]
            plt.figure()
            sns.histplot(df[temp_col], kde=True)
            plt.title(f'Temperature Distribution - {room_name}')
            plt.xlabel('Temperature')
            plt.ylabel('Frequency')
            plt.tight_layout()
            plt.show()
        else:
            print(f"No temperature column found in {file}")
            
    except Exception as e:
        print(f"Error processing {file}: {e}")

## Box plots

In [ ]:
# Set larger figure size for better visualization
plt.rcParams['figure.figsize'] = (12, 6)

# List of files to process
files = [
    'Apartment_1_Kitchen_motion_data.csv',
    'Apartment_1_Office_motion_data.csv', 
    'Apartment_2_Living_motion_data.csv',
    'Apartment_2_Office_motion_data.csv'
]

# Process each file
for file in files:
    try:
        # Load the data
        df = pd.read_csv(file)
        apt_num = file.split('_')[1]  # Extract apartment number
        room_name = file.split('_')[3]  # Extract room name
        title_prefix = f"Apartment {apt_num} {room_name}"
        
        print(f"\n=== Processing {title_prefix} data ===")
        print(f"Columns found: {df.columns.tolist()}")
        
        # Find the light and temperature columns
        light_col = next((col for col in df.columns if 'light' in col.lower()), None)
        temp_col = next((col for col in df.columns if 'temp' in col.lower()), None)
        
        # 1. For Light column
        if light_col:
            plt.figure()
            sns.boxplot(y=df[light_col])
            plt.title(f'Light Level Distribution - {title_prefix}')
            plt.ylabel('Light Level')
            # Add text with statistics
            stats = f"Min: {df[light_col].min():.2f}\nMax: {df[light_col].max():.2f}\nMean: {df[light_col].mean():.2f}\nMedian: {df[light_col].median():.2f}"
            plt.text(0.05, 0.95, stats, transform=plt.gca().transAxes, 
                     verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
            plt.tight_layout()
            plt.show()
        else:
            print(f"No light column found in {file}")
        
        # 2. For Temperature column
        if temp_col:
            plt.figure()
            sns.boxplot(y=df[temp_col])
            plt.title(f'Temperature Distribution - {title_prefix}')
            plt.ylabel('Temperature')
            # Add text with statistics
            stats = f"Min: {df[temp_col].min():.2f}\nMax: {df[temp_col].max():.2f}\nMean: {df[temp_col].mean():.2f}\nMedian: {df[temp_col].median():.2f}"
            plt.text(0.05, 0.95, stats, transform=plt.gca().transAxes, 
                     verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
            plt.tight_layout()
            plt.show()
        else:
            print(f"No temperature column found in {file}")
            
    except Exception as e:
        print(f"Error processing {file}: {e}")

## Process and Remove columns

In [ ]:
def process_columns(csv_file, output_file=None):

    # Read the CSV file
    df = pd.read_csv(csv_file)
    
    # Convert DateTime to datetime type
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    
    # Extract minutes from Hours column (format HH:MM)
    df['Minute'] = df['Hours'].str.split(':', expand=True)[1].astype(int)
    
    # Rearrange columns to put Minute right after Hour
    hour_pos = df.columns.get_loc('Hour')
    cols = list(df.columns)
    minute_pos = cols.index('Minute')
    cols.pop(minute_pos)  # Remove Minute from its current position
    cols.insert(hour_pos + 1, 'Minute')  # Insert Minute after Hour
    
    # Drop columns
    cols.remove('Hours')
    cols.remove('EntryCounter')
    cols.remove('Date')
    cols.remove('ApartmentID')
    cols.remove('User')
    
    # Reorder the DataFrame
    df = df[cols]
    
    # If an output file is specified, save the result
    if output_file:
        df.to_csv(output_file, index=False)
        print(f"Processed data saved to {output_file}")
        return None
    
    # Otherwise, return the processed DataFrame
    return df

input_files = [
    'Apartment_1_Kitchen_motion_data.csv',
    'Apartment_1_Office_motion_data.csv', 
    'Apartment_2_Living_motion_data.csv',
    'Apartment_2_Office_motion_data.csv'
]

for input_file in input_files:
    process_columns(input_file, input_file)
    print(f"Processed {input_file}")


## Feature Engineering: Presence features and attribute Presence

In [ ]:
def create_presence_features(df, room_name):

    # Create a copy to avoid modifying the original
    room_df = df.copy()
    
    # Ensure DataFrame has a datetime index (if it doesn't already)
    if not isinstance(room_df.index, pd.DatetimeIndex) and 'DateTime' in room_df.columns:
        room_df = room_df.set_index('DateTime')
        room_df.index = pd.to_datetime(room_df.index)
    
    # Extract date only for grouping
    date_only = room_df.index.date if isinstance(room_df.index, pd.DatetimeIndex) else pd.to_datetime(room_df['Date']).dt.date
    
    # Create motion-related features
    motion_col = f"{room_name}_Motion_Motion"
    
    # 2. Create lagged features to capture transitions
    for lag in [1, 3, 5]:
        room_df[f'{room_name}_motion_lag_{lag}'] = room_df.groupby(date_only)[motion_col].shift(lag, fill_value=0)
    
    # 3. Create forward-looking features to capture future actions
    for lead in [1, 3, 5]:
        room_df[f'{room_name}_motion_lead_{lead}'] = room_df.groupby(date_only)[motion_col].shift(-lead, fill_value=0)
    
    return room_df

def label_presence(df, room_name):

    # Create a copy to avoid modifying the original
    labeled_df = df.copy()
    
    # Get the motion column name
    motion_col = f"{room_name}_Motion_Motion"
    
    # Create a new column for presence
    labeled_df[f'{room_name}_Presence'] = 0
    
    # If motion is detected directly, set presence to 1
    labeled_df.loc[labeled_df[motion_col] == 1, f'{room_name}_Presence'] = 1
    
    # Use lag and lead pairs to detect gaps (0s) between motion detections (1s)
    # This ensures we only fill gaps where there is motion both before AND after
    for lag in [1, 3, 5]:
        for lead in [1, 3, 5]:
            lag_col = f'{room_name}_motion_lag_{lag}'
            lead_col = f'{room_name}_motion_lead_{lead}'
            
            # Mark as presence only if there's motion both before AND after
            labeled_df.loc[
                (labeled_df[motion_col] == 0) & 
                (labeled_df[lag_col] == 1) & 
                (labeled_df[lead_col] == 1), 
                f'{room_name}_Presence'
            ] = 1
    
    
    return labeled_df

    
# Load data for both apartments
apt1_kitchen = pd.read_csv('Apartment_1_Kitchen_motion_data.csv')
apt1_office = pd.read_csv('Apartment_1_Office_motion_data.csv')
apt2_living = pd.read_csv('Apartment_2_Living_motion_data.csv')
apt2_office = pd.read_csv('Apartment_2_Office_motion_data.csv')

# Convert DateTime to datetime for proper processing
for df in [apt1_kitchen, apt1_office, apt2_living, apt2_office]:
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df.set_index('DateTime', inplace=True)
    df.sort_index(inplace=True)

# Process each room with feature engineering
processed_apt1_kitchen = create_presence_features(apt1_kitchen, 'Kitchen')
processed_apt1_office = create_presence_features(apt1_office, 'Office')
processed_apt2_living = create_presence_features(apt2_living, 'Livingroom')
processed_apt2_office = create_presence_features(apt2_office, 'Office')

# Label presence for each room
labeled_apt1_kitchen = label_presence(processed_apt1_kitchen, 'Kitchen')
labeled_apt1_office = label_presence(processed_apt1_office, 'Office')
labeled_apt2_living = label_presence(processed_apt2_living, 'Livingroom')
labeled_apt2_office = label_presence(processed_apt2_office, 'Office')

# Save labeled data
labeled_apt1_kitchen.to_csv('Apartment_1_Kitchen_motion_data.csv')
labeled_apt1_office.to_csv('Apartment_1_Office_motion_data.csv')
labeled_apt2_living.to_csv('Apartment_2_Living_motion_data.csv')
labeled_apt2_office.to_csv('Apartment_2_Office_motion_data.csv')


## Aggregate to hourly data

In [ ]:
def aggregate_presence_to_hourly(filepath, room_name):

    # Load the labeled dataset
    df = pd.read_csv(filepath)
    
    # Convert DateTime to proper datetime format if it's not already
    if 'DateTime' in df.columns:
        df['DateTime'] = pd.to_datetime(df['DateTime'])
    else:
        # If DateTime is the index
        df.index = pd.to_datetime(df.index)
        df = df.reset_index()
    
    # Create a Date column for grouping
    df['Date'] = df['DateTime'].dt.date
    
    # Drop lag and lead columns as they're no longer needed
    lag_lead_cols = [col for col in df.columns if 'lag' in col or 'lead' in col]
    df = df.drop(columns=lag_lead_cols)
    
    # Define columns
    motion_col = f"{room_name}_Motion_Motion"
    presence_col = f"{room_name}_Presence"
    temp_col = f"{room_name}_Motion_Temperature"
    light_col = f"{room_name}_Motion_Light"
    
    # Prepare aggregation dictionary - correct format
    agg_dict = {
        'DayOfWeek': 'first',
        'IsWeekend': 'first',
        motion_col: 'sum',  # Sum of motion events
        presence_col: 'sum'  # Sum of presence minutes
    }
    
    # Add temperature and light columns if they exist
    if temp_col in df.columns:
        agg_dict[temp_col] = 'mean'
    if light_col in df.columns:
        agg_dict[light_col] = 'mean'
    
    # Add entry count using a different approach
    df['EntryCount'] = 1  # Each row is one entry
    agg_dict['EntryCount'] = 'sum'
    
    # Group by date and hour
    hourly_data = df.groupby(['Date', 'Hour']).agg(agg_dict).reset_index()
    
    # Rename columns to reflect the aggregation method
    column_renames = {
        motion_col: f"{motion_col}_Sum",
        presence_col: f"{presence_col}_Sum"
    }
    
    if temp_col in hourly_data.columns:
        column_renames[temp_col] = f"{temp_col}_Mean"
    if light_col in hourly_data.columns:
        column_renames[light_col] = f"{light_col}_Mean"
    
    hourly_data = hourly_data.rename(columns=column_renames)
    
    # Format Hours as HH:00
    hourly_data['Hours'] = hourly_data['Hour'].apply(lambda x: f"{x:02d}:00")
    
    # Rename Date column to DateTime to maintain format from original script
    hourly_data = hourly_data.rename(columns={'Date': 'DateTime'})
    
    # Calculate presence percentage for the hour
    hourly_data[f'{room_name}_Presence_Percentage'] = (hourly_data[f"{presence_col}_Sum"] / hourly_data['EntryCount'] * 100).round(2)
    
    # Reorder columns to have DateTime and Hours at the beginning
    base_columns = ['DateTime', 'Hours', 'Hour', 'DayOfWeek', 'IsWeekend']
    data_columns = [col for col in hourly_data.columns if col not in base_columns + ['EntryCount']]
    columns_order = base_columns + data_columns + ['EntryCount']
    hourly_data = hourly_data[columns_order]
    
    # Check for hours with fewer than expected entries
    expected_entries = 60  # 60 minutes per hour
    missing_data = hourly_data[hourly_data['EntryCount'] < expected_entries]
    print(f"\nHours with fewer than {expected_entries} entries (potentially missing data):")
    print(missing_data[['DateTime', 'Hours', 'EntryCount']])
    
    # Calculate average entries per hour
    avg_entries = hourly_data['EntryCount'].mean()
    print(f"\nAverage entries per hour: {avg_entries:.2f}")
    
    return hourly_data

# Process each room's data
rooms = {
    'Apartment_1_Kitchen_motion_data.csv': 'Kitchen',
    'Apartment_1_Office_motion_data.csv': 'Office',
    'Apartment_2_Living_motion_data.csv': 'Livingroom',
    'Apartment_2_Office_motion_data.csv': 'Office'
}

for filepath, room_name in rooms.items():
    print(f"Processing {room_name} data from {filepath}")
    hourly_data = aggregate_presence_to_hourly(filepath, room_name)
    
    # Save the hourly aggregated data
    output_filepath = f"{filepath}"
    hourly_data.to_csv(output_filepath, index=False)
    print(f"Hourly aggregated data saved to '{output_filepath}'")
    print(f"First few rows of {room_name} hourly data:")
    print(hourly_data.head(5))
    print("-" * 80)

## Visualization: Presence data

In [ ]:
def visualize_presence_data(filepath, room_name):

    # Load the data
    df = pd.read_csv(filepath)
    
    # Convert DateTime to datetime format
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    
    # Create a proper timestamp combining date and hour
    df['Timestamp'] = pd.to_datetime(df['DateTime'].astype(str) + ' ' + df['Hours'])
    
    # Define column names based on room_name
    motion_sum_col = f"{room_name}_Motion_Motion_Sum"
    presence_sum_col = f"{room_name}_Presence_Sum"
    presence_pct_col = f"{room_name}_Presence_Percentage"
    temp_mean_col = f"{room_name}_Motion_Temperature_Mean" if f"{room_name}_Motion_Temperature_Mean" in df.columns else None
    light_mean_col = f"{room_name}_Motion_Light_Mean" if f"{room_name}_Motion_Light_Mean" in df.columns else None
    
    # Set better column names for plotting
    column_mapping = {
        motion_sum_col: f'{room_name} Motion Events',
        presence_sum_col: f'{room_name} Presence (minutes)',
        presence_pct_col: f'{room_name} Presence (%)'
    }
    
    if temp_mean_col:
        column_mapping[temp_mean_col] = f'{room_name} Temperature (°C)'
    if light_mean_col:
        column_mapping[light_mean_col] = f'{room_name} Light Level'
    
    # Create shorter names for the variables
    df = df.rename(columns=column_mapping)
    
    # 1. Hourly pattern visualization (average by hour of day)
    plt.figure(figsize=(15, 12))
    
    # Number of subplots depends on available data
    num_plots = 2  # Motion and Presence always available
    if temp_mean_col:
        num_plots += 1
    if light_mean_col:
        num_plots += 1
    
    # Motion events by hour of day
    plt.subplot(num_plots, 1, 1)
    hourly_motion = df.groupby('Hour')[f'{room_name} Motion Events'].mean()
    sns.barplot(x=hourly_motion.index, y=hourly_motion.values)
    plt.title(f'Average {room_name} Motion Events by Hour of Day')
    plt.xlabel('Hour')
    plt.ylabel('Motion Events')
    plt.xticks(range(0, 24, 2))
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Presence by hour of day
    plt.subplot(num_plots, 1, 2)
    hourly_presence = df.groupby('Hour')[f'{room_name} Presence (%)'].mean()
    sns.barplot(x=hourly_presence.index, y=hourly_presence.values)
    plt.title(f'Average {room_name} Presence % by Hour of Day')
    plt.xlabel('Hour')
    plt.ylabel('Presence (%)')
    plt.xticks(range(0, 24, 2))
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Temperature by hour of day (if available)
    plot_idx = 3
    if temp_mean_col:
        plt.subplot(num_plots, 1, plot_idx)
        hourly_temp = df.groupby('Hour')[f'{room_name} Temperature (°C)'].mean()
        sns.barplot(x=hourly_temp.index, y=hourly_temp.values)
        plt.title(f'Average {room_name} Temperature by Hour of Day')
        plt.xlabel('Hour')
        plt.ylabel('Temperature (°C)')
        plt.xticks(range(0, 24, 2))
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plot_idx += 1
    
    # Light level by hour of day (if available)
    if light_mean_col:
        plt.subplot(num_plots, 1, plot_idx)
        hourly_light = df.groupby('Hour')[f'{room_name} Light Level'].mean()
        sns.barplot(x=hourly_light.index, y=hourly_light.values)
        plt.title(f'Average {room_name} Light Level by Hour of Day')
        plt.xlabel('Hour')
        plt.ylabel('Light Level')
        plt.xticks(range(0, 24, 2))
        plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()
    
    # 2. Daily time series visualization
    # Aggregate data by day
    agg_dict = {
        f'{room_name} Motion Events': 'sum',
        f'{room_name} Presence (minutes)': 'sum',
        f'{room_name} Presence (%)': 'mean'
    }
    
    if temp_mean_col:
        agg_dict[f'{room_name} Temperature (°C)'] = 'mean'
    if light_mean_col:
        agg_dict[f'{room_name} Light Level'] = 'mean'
    
    daily_df = df.groupby(df['DateTime'].dt.date).agg(agg_dict).reset_index()
    daily_df['DateTime'] = pd.to_datetime(daily_df['DateTime'])
    
    plt.figure(figsize=(15, 15))
    
    # Daily motion events
    plt.subplot(num_plots, 1, 1)
    plt.plot(daily_df['DateTime'], daily_df[f'{room_name} Motion Events'], marker='o', linestyle='-')
    plt.title(f'Daily {room_name} Motion Events')
    plt.ylabel('Motion Events')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(rotation=45)
    
    # Daily presence
    plt.subplot(num_plots, 1, 2)
    plt.plot(daily_df['DateTime'], daily_df[f'{room_name} Presence (%)'], marker='o', linestyle='-', color='green')
    plt.title(f'Daily Average {room_name} Presence %')
    plt.ylabel('Presence (%)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(rotation=45)
    
    # Daily temperature (if available)
    plot_idx = 3
    if temp_mean_col:
        plt.subplot(num_plots, 1, plot_idx)
        plt.plot(daily_df['DateTime'], daily_df[f'{room_name} Temperature (°C)'], marker='o', linestyle='-', color='red')
        plt.title(f'Daily Average {room_name} Temperature')
        plt.ylabel('Temperature (°C)')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.xticks(rotation=45)
        plot_idx += 1
    
    # Daily light level (if available)
    if light_mean_col:
        plt.subplot(num_plots, 1, plot_idx)
        plt.plot(daily_df['DateTime'], daily_df[f'{room_name} Light Level'], marker='o', linestyle='-', color='orange')
        plt.title(f'Daily Average {room_name} Light Level')
        plt.ylabel('Light Level')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # 3. Weekly visualization (to see patterns more clearly)
    # Select a week of data for detailed visualization
    # First get the first complete week in the dataset
    start_date = df['DateTime'].min()
    # Find the next Monday
    while start_date.dayofweek != 0:  # 0 is Monday
        start_date += pd.Timedelta(days=1)
    
    end_date = start_date + pd.Timedelta(days=6)
    week_data = df[(df['DateTime'] >= start_date) & (df['DateTime'] <= end_date)]
    
    if len(week_data) > 0:  # Only proceed if we have a complete week
        plt.figure(figsize=(15, 12))
        
        # Motion events for one week
        plt.subplot(num_plots, 1, 1)
        plt.plot(week_data['Timestamp'], week_data[f'{room_name} Motion Events'], marker='.', linestyle='-')
        plt.title(f'{room_name} Motion Events (One Week)')
        plt.ylabel('Motion Events')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.xticks(rotation=45)
        
        # Presence for one week
        plt.subplot(num_plots, 1, 2)
        plt.plot(week_data['Timestamp'], week_data[f'{room_name} Presence (%)'], marker='.', linestyle='-', color='green')
        plt.title(f'{room_name} Presence % (One Week)')
        plt.ylabel('Presence (%)')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.xticks(rotation=45)
        
        # Temperature for one week (if available)
        plot_idx = 3
        if temp_mean_col:
            plt.subplot(num_plots, 1, plot_idx)
            plt.plot(week_data['Timestamp'], week_data[f'{room_name} Temperature (°C)'], marker='.', linestyle='-', color='red')
            plt.title(f'{room_name} Temperature (One Week)')
            plt.ylabel('Temperature (°C)')
            plt.grid(True, linestyle='--', alpha=0.7)
            plt.xticks(rotation=45)
            plot_idx += 1
        
        # Light level for one week (if available)
        if light_mean_col:
            plt.subplot(num_plots, 1, plot_idx)
            plt.plot(week_data['Timestamp'], week_data[f'{room_name} Light Level'], marker='.', linestyle='-', color='orange')
            plt.title(f'{room_name} Light Level (One Week)')
            plt.ylabel('Light Level')
            plt.grid(True, linestyle='--', alpha=0.7)
            plt.xticks(rotation=45)
        
        plt.tight_layout()
        plt.show()
    else:
        print(f"Not enough data for a complete week visualization for {room_name}")
    
    # 4. Correlation visualization
    corr_columns = [f'{room_name} Motion Events', f'{room_name} Presence (%)']
    
    if temp_mean_col:
        corr_columns.append(f'{room_name} Temperature (°C)')
    if light_mean_col:
        corr_columns.append(f'{room_name} Light Level')
    
    if len(corr_columns) > 2:  # Only do correlation if we have more than motion and presence
        correlation_df = df[corr_columns]
        correlation_matrix = correlation_df.corr()
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
        plt.title(f'Correlation Between {room_name} Presence and Other Variables')
        plt.tight_layout()
        plt.show()
    
    # 5. Weekday vs Weekend Comparison
    plt.figure(figsize=(12, 8))
    
    # For presence
    plt.subplot(2, 1, 1)
    weekday_hourly = df[df['IsWeekend'] == 0].groupby('Hour')[f'{room_name} Presence (%)'].mean()
    weekend_hourly = df[df['IsWeekend'] == 1].groupby('Hour')[f'{room_name} Presence (%)'].mean()
    
    plt.plot(weekday_hourly.index, weekday_hourly.values, marker='o', linestyle='-', label='Weekday')
    plt.plot(weekend_hourly.index, weekend_hourly.values, marker='s', linestyle='--', label='Weekend')
    plt.title(f'{room_name} Presence % by Hour: Weekday vs Weekend')
    plt.xlabel('Hour')
    plt.ylabel('Presence (%)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(range(0, 24, 2))
    
    # For motion
    plt.subplot(2, 1, 2)
    weekday_motion = df[df['IsWeekend'] == 0].groupby('Hour')[f'{room_name} Motion Events'].mean()
    weekend_motion = df[df['IsWeekend'] == 1].groupby('Hour')[f'{room_name} Motion Events'].mean()
    
    plt.plot(weekday_motion.index, weekday_motion.values, marker='o', linestyle='-', label='Weekday')
    plt.plot(weekend_motion.index, weekend_motion.values, marker='s', linestyle='--', label='Weekend')
    plt.title(f'{room_name} Motion Events by Hour: Weekday vs Weekend')
    plt.xlabel('Hour')
    plt.ylabel('Motion Events')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xticks(range(0, 24, 2))
    
    plt.tight_layout()
    plt.show()

# Process each room's data
rooms = {
    'Apartment_1_Kitchen_motion_data.csv': 'Kitchen',
    'Apartment_1_Office_motion_data.csv': 'Office',
    'Apartment_2_Living_motion_data.csv': 'Livingroom',
    'Apartment_2_Office_motion_data.csv': 'Office'
}

for filepath, room_name in rooms.items():
    print(f"Creating visualizations for {room_name} data from {filepath}")
    visualize_presence_data(filepath, room_name)
    print(f"Completed visualizations for {room_name}")
    print("-" * 80)

## Test stationarity

In [ ]:
def test_stationarity(series, title, significance=0.05):
    
    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Plot the time series
    ax1.plot(series, label='Original')
    ax1.set_title(f'Time Series Analysis: {title}')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Value')
    
    # Calculate and plot rolling statistics
    rolling_mean = series.rolling(window=24).mean()  # 24-hour window
    rolling_std = series.rolling(window=24).std()
    
    ax1.plot(rolling_mean, label='Rolling Mean (24h)', color='red')
    ax1.plot(rolling_std, label='Rolling Std (24h)', color='green')
    ax1.legend(loc='best')
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # Perform Augmented Dickey-Fuller test
    result = adfuller(series.dropna())
    
    # Extract and format ADF test results
    adf_stat = result[0]
    p_value = result[1]
    critical_values = result[4]
    
    # Create text for ADF test results
    adf_text = f'ADF Statistic: {adf_stat:.4f}\n'
    adf_text += f'p-value: {p_value:.4f}\n'
    adf_text += 'Critical Values:\n'
    for key, value in critical_values.items():
        adf_text += f'   {key}: {value:.4f}\n'
    
    # Highlight stationarity result
    is_stationary = p_value < significance
    result_text = f'Result: The series is {"stationary" if is_stationary else "non-stationary"}'
    adf_text += f'\n{result_text}'
    
    # Plot autocorrelation
    plot_acf(series.dropna(), ax=ax2, lags=48)  # Show 48 lags (2 days for hourly data)
    ax2.set_title('Autocorrelation Function')
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    # Add ADF test results as text
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    fig.text(0.15, 0.15, adf_text, fontsize=12, bbox=props)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to make room for text
    plt.show()
    
    # Create a separate plot for PACF
    plt.figure(figsize=(12, 4))
    plot_pacf(series.dropna(), lags=48)
    plt.title(f'Partial Autocorrelation Function: {title}')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    return is_stationary, p_value, adf_stat

# Function to decompose time series
def decompose_series(series, title, period=24):

    try:
        # Decompose the series (additive model)
        decomposition = seasonal_decompose(series, model='additive', period=period)
        
        plt.figure(figsize=(14, 12))
        plt.subplot(411)
        plt.plot(decomposition.observed)
        plt.title(f'Original Series: {title}')
        plt.grid(True, linestyle='--', alpha=0.5)
        
        plt.subplot(412)
        plt.plot(decomposition.trend)
        plt.title('Trend Component')
        plt.grid(True, linestyle='--', alpha=0.5)
        
        plt.subplot(413)
        plt.plot(decomposition.seasonal)
        plt.title(f'Seasonal Component (period={period})')
        plt.grid(True, linestyle='--', alpha=0.5)
        
        plt.subplot(414)
        plt.plot(decomposition.resid)
        plt.title('Residual Component')
        plt.grid(True, linestyle='--', alpha=0.5)
        
        plt.tight_layout()
        plt.show()
        
        # Test if residuals are stationary
        print("\nTesting stationarity of the residual component...")
        test_stationarity(decomposition.resid.dropna(), f'Residual Component of {title}')
        
        return decomposition
        
    except Exception as e:
        print(f"Could not perform seasonal decomposition: {e}")
        print("This might be due to missing values or insufficient data.")
        return None

def analyze_room_presence(filepath, room_name):

    print(f"\n{'='*80}")
    print(f"Analyzing {room_name} Presence Data")
    print(f"{'='*80}")
    
    # Load the data
    df = pd.read_csv(filepath)
    
    # Convert DateTime to datetime format
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    
    # Set DateTime as index
    df_indexed = df.set_index('DateTime')
    
    # Define columns for analysis
    motion_col = f"{room_name}_Motion_Motion_Sum"
    presence_col = f"{room_name}_Presence_Sum"
    presence_pct_col = f"{room_name}_Presence_Percentage"
    
    # Test stationarity for Motion data
    print(f"\nTesting stationarity of {room_name} Motion Events...")
    motion_stationary, motion_p, _ = test_stationarity(
        df_indexed[motion_col], 
        f'{room_name} Motion Events'
    )
    
    # Test stationarity for Presence percentage
    print(f"\nTesting stationarity of {room_name} Presence Percentage...")
    presence_stationary, presence_p, _ = test_stationarity(
        df_indexed[presence_pct_col], 
        f'{room_name} Presence Percentage'
    )
    
    # If motion data is non-stationary, test differences
    if not motion_stationary:
        print(f"\nMotion data is non-stationary (p-value: {motion_p:.4f}). Testing differences...")
        
        # First difference
        print("\nTesting first difference of motion data...")
        diff1 = df_indexed[motion_col].diff().dropna()
        test_stationarity(diff1, f'First Difference of {room_name} Motion Events')
        
        # Seasonal difference (24-hour for daily patterns)
        print("\nTesting seasonal difference (24-hour) of motion data...")
        if len(df_indexed) >= 25:  # Need at least 25 points for a 24-hour difference
            seasonal_diff = df_indexed[motion_col].diff(24).dropna()
            test_stationarity(seasonal_diff, f'Seasonal Difference (24h) of {room_name} Motion Events')
            
            # Both differences
            print("\nTesting both first and seasonal difference of motion data...")
            diff_both = df_indexed[motion_col].diff().diff(24).dropna()
            test_stationarity(diff_both, f'First and Seasonal Difference of {room_name} Motion Events')
        else:
            print("Not enough data points for seasonal differencing.")
    
    # If presence data is non-stationary, test differences
    if not presence_stationary:
        print(f"\nPresence data is non-stationary (p-value: {presence_p:.4f}). Testing differences...")
        
        # First difference
        print("\nTesting first difference of presence data...")
        diff1 = df_indexed[presence_pct_col].diff().dropna()
        test_stationarity(diff1, f'First Difference of {room_name} Presence Percentage')
        
        # Seasonal difference (24-hour for daily patterns)
        print("\nTesting seasonal difference (24-hour) of presence data...")
        if len(df_indexed) >= 25:  # Need at least 25 points for a 24-hour difference
            seasonal_diff = df_indexed[presence_pct_col].diff(24).dropna()
            test_stationarity(seasonal_diff, f'Seasonal Difference (24h) of {room_name} Presence Percentage')
            
            # Both differences
            print("\nTesting both first and seasonal difference of presence data...")
            diff_both = df_indexed[presence_pct_col].diff().diff(24).dropna()
            test_stationarity(diff_both, f'First and Seasonal Difference of {room_name} Presence Percentage')
        else:
            print("Not enough data points for seasonal differencing.")
    
    # Decompose time series (if enough data)
    if len(df_indexed) >= 2*24:  # Need at least 2 days of data for meaningful decomposition
        print("\nPerforming seasonal decomposition of motion data...")
        decompose_series(df_indexed[motion_col], f'{room_name} Motion Events')
        
        print("\nPerforming seasonal decomposition of presence data...")
        decompose_series(df_indexed[presence_pct_col], f'{room_name} Presence Percentage')
    else:
        print("\nNot enough data for seasonal decomposition (need at least 48 hours).")
    
    print(f"\nCompleted analysis for {room_name}\n")

# Process each room's data
rooms = {
    'Apartment_1_Kitchen_motion_data.csv': 'Kitchen',
    'Apartment_1_Office_motion_data.csv': 'Office',
    'Apartment_2_Living_motion_data.csv': 'Livingroom',
    'Apartment_2_Office_motion_data.csv': 'Office'
}

for filepath, room_name in rooms.items():
    try:
        analyze_room_presence(filepath, room_name)
    except Exception as e:
        print(f"Error analyzing {room_name} data: {e}")

## Define Features and Target

In [ ]:
#Refactored
def prepare_dataset(file_path, room_prefix):

    # Load the feature-engineered dataset
    df = pd.read_csv(file_path)
    
    # Convert DateTime to proper datetime format for feature extraction
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    
    # Extract additional time components
    df['Year'] = df['DateTime'].dt.year
    df['Month'] = df['DateTime'].dt.month
    df['Day'] = df['DateTime'].dt.day
    df['DayOfYear'] = df['DateTime'].dt.dayofyear
    
    # Define the target variable
    target = f'{room_prefix}_Presence_Sum'
    
    # Define non-feature columns
    non_features = [
        'DateTime',       # Date string
        'Hours',          # Time string            
        'EntryCount',
        f'{room_prefix}_Presence_Sum', # Target variable
        f'{room_prefix}_Presence_Percentage'
    ]
    
    # Define feature columns
    features = [
        'Hour', 'DayOfWeek', 'IsWeekend', 'Month', 'Year', 'Day', 'DayOfYear',
        f'{room_prefix}_Motion_Motion_Sum',
        f'{room_prefix}_Motion_Temperature_Mean',
        f'{room_prefix}_Motion_Light_Mean'
    ]
    
    return df, target, non_features, features

# Create all original variable names using a function to reduce repetition
def process_apartment_room(file_path, apartment_number, room_name):

    # Load the dataset
    df = pd.read_csv(file_path)
    
    # Convert DateTime to proper datetime format
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    
    # Extract time components
    df['Year'] = df['DateTime'].dt.year
    df['Month'] = df['DateTime'].dt.month
    df['Day'] = df['DateTime'].dt.day
    df['DayOfYear'] = df['DateTime'].dt.dayofyear
    
    # Create variable names as they were in original code
    var_prefix = f"app{apartment_number}_{room_name.lower()}"
    
    # Set the variables using globals() to create variables in global scope
    target_var_name = f"target_{var_prefix}"
    non_features_var_name = f"non_features_{var_prefix}"
    features_var_name = f"features_{var_prefix}"
    
    # Define variable values
    target_value = f"{room_name}_Presence_Sum"
    
    non_features_value = [
        'DateTime',
        'Hours',
        'EntryCount',
        f'{room_name}_Presence_Sum',
        f'{room_name}_Presence_Percentage'
    ]
    
    features_value = [
        'Hour', 'DayOfWeek', 'IsWeekend', 'Month', 'Year', 'Day', 'DayOfYear',
        f'{room_name}_Motion_Motion_Sum',
        f'{room_name}_Motion_Temperature_Mean',
        f'{room_name}_Motion_Light_Mean'
    ]
    
    # Return a dictionary with all the variables and their values
    return {
        'df': df,
        target_var_name: target_value,
        non_features_var_name: non_features_value,
        features_var_name: features_value
    }

# Process each apartment-room combination
# Apartment 1 Kitchen
result_app1_kitchen = process_apartment_room('Apartment_1_Kitchen_motion_data.csv', 1, 'Kitchen')
df_app1_kitchen = result_app1_kitchen['df']
target_app1_kitchen = result_app1_kitchen['target_app1_kitchen']
non_features_app1_kitchen = result_app1_kitchen['non_features_app1_kitchen']
features_app1_kitchen = result_app1_kitchen['features_app1_kitchen']

# Apartment 1 Office
result_app1_office = process_apartment_room('Apartment_1_Office_motion_data.csv', 1, 'Office')
df_app1_office = result_app1_office['df']
target_app1_office = result_app1_office['target_app1_office']
non_features_app1_office = result_app1_office['non_features_app1_office']
features_app1_office = result_app1_office['features_app1_office']

# Apartment 2 Living Room
result_app2_living = process_apartment_room('Apartment_2_Living_motion_data.csv', 2, 'Livingroom')
df_app2_living = result_app2_living['df']
target_app2_living = result_app2_living['target_app2_livingroom']
non_features_app2_living = result_app2_living['non_features_app2_livingroom']
features_app2_living = result_app2_living['features_app2_livingroom']

# Apartment 2 Office
result_app2_office = process_apartment_room('Apartment_2_Office_motion_data.csv', 2, 'Office')
df_app2_office = result_app2_office['df']
target_app2_office = result_app2_office['target_app2_office']
non_features_app2_office = result_app2_office['non_features_app2_office']
features_app2_office = result_app2_office['features_app2_office']

## Train and Test split

In [ ]:
def create_train_test_split(df, features, target, split_percentage=0.2):

    # Make sure we have datetime format
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    
    # Sort the dataframe by DateTime to ensure chronological order
    df = df.sort_values('DateTime')
    
    # Display date range of the dataset
    print(f"Dataset spans from {df['DateTime'].min()} to {df['DateTime'].max()}")
    print(f"Total number of data points: {len(df)}")
    
    # Calculate the split point - using the last 20% of the data for testing
    split_index = int(len(df) * (1 - split_percentage))
    
    # Split the data chronologically
    train_df = df.iloc[:split_index]
    test_df = df.iloc[split_index:]
    
    # Split into features and target
    X_train = train_df[features]
    y_train = train_df[target]
    X_test = test_df[features]
    y_test = test_df[target]
    
    print(f"Training set size: {len(train_df)} samples")
    print(f"Testing set size: {len(test_df)} samples")
    
    return X_train, y_train, X_test, y_test, train_df, test_df

# Create train/test splits for each dataset
print("Creating train/test split for Apartment 1 Kitchen:")
X_train_app1_kitchen, y_train_app1_kitchen, X_test_app1_kitchen, y_test_app1_kitchen, train_df_app1_kitchen, test_df_app1_kitchen = create_train_test_split(
    df_app1_kitchen, features_app1_kitchen, target_app1_kitchen
)

print("\nCreating train/test split for Apartment 1 Office:")
X_train_app1_office, y_train_app1_office, X_test_app1_office, y_test_app1_office, train_df_app1_office, test_df_app1_office = create_train_test_split(
    df_app1_office, features_app1_office, target_app1_office
)

print("\nCreating train/test split for Apartment 2 Living Room:")
X_train_app2_living, y_train_app2_living, X_test_app2_living, y_test_app2_living, train_df_app2_living, test_df_app2_living = create_train_test_split(
    df_app2_living, features_app2_living, target_app2_living
)

print("\nCreating train/test split for Apartment 2 Office:")
X_train_app2_office, y_train_app2_office, X_test_app2_office, y_test_app2_office, train_df_app2_office, test_df_app2_office = create_train_test_split(
    df_app2_office, features_app2_office, target_app2_office
)

## Train-Test: Random forest

In [ ]:
def train_test_random_forest(X_train, y_train, X_test, y_test, features, dataset_name):
    """
    Train a Random Forest regressor on the given dataset and evaluate performance.
    """
    print(f"\n{'='*80}")
    print(f"Training Random Forest Model for {dataset_name}")
    print(f"{'='*80}")
    
    # Initialize the Random Forest model
    rf_model = RandomForestRegressor(
        n_estimators=150,            # Increased number of trees for more stability
        criterion='squared_error',   # Standard criterion for regression
        max_depth=15,                # Limit depth to prevent overfitting
        min_samples_split=5,         # Require more samples to split a node
        min_samples_leaf=4,          # Require more samples in leaf nodes
        max_features=0.7,            # Use 70% of features at each split
        bootstrap=True,              # Use bootstrap samples
        oob_score=True,              # Calculate out-of-bag score for better validation
        n_jobs=-1,                   # Use all cores
        random_state=42              # For reproducibility
    )
    
    # Train the model
    print("Training the model...")
    rf_model.fit(X_train, y_train)
    
    # Out-of-bag score (a form of cross-validation)
    print(f"Out-of-bag score: {rf_model.oob_score_:.4f}")
    
    # Make predictions on training and test sets
    y_train_pred = rf_model.predict(X_train)
    y_test_pred = rf_model.predict(X_test)
    
    # Evaluate the model
    print("\nModel Evaluation:")
    print("-" * 50)
    
    # Calculate metrics for both sets
    train_metrics = calculate_metrics(y_train, y_train_pred, "Training")
    test_metrics = calculate_metrics(y_test, y_test_pred, "Test")
    
    # Feature importance
    print("\nFeature Importance:")
    print("-" * 50)
    
    # Get feature importances
    feature_importances = rf_model.feature_importances_
    features_df = pd.DataFrame({
        'Feature': features,
        'Importance': feature_importances
    }).sort_values('Importance', ascending=False)
    
    # Display top features
    print(features_df.head(10))
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    top_features = features_df.head(10)
    sns.barplot(x='Importance', y='Feature', data=top_features)
    plt.title(f'Top 10 Feature Importance for {dataset_name}')
    plt.tight_layout()
    plt.show()
    
    # Return model and metrics
    metrics = {
        'train': train_metrics,
        'test': test_metrics,
        'feature_importance': features_df
    }
    
    return rf_model, metrics

def calculate_metrics(y_true, y_pred, set_name):
    """
    Calculate regression performance metrics
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # Alternative MAPE calculation that excludes zeros and is capped at 100%
    non_zero_mask = np.abs(y_true) > 0.1  # Only include values > 0.1
    if np.sum(non_zero_mask) > 0:
        mape_values = np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask]) * 100
        mape_capped = np.minimum(mape_values, 100)  # Cap at 100%
        mape = np.mean(mape_capped)
    else:
        mape = np.nan  # Not applicable if no non-zero values
    
    print(f"{set_name} Set Metrics:")
    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    if not np.isnan(mape):
        print(f"Modified MAPE (non-zero values only, capped): {mape:.2f}%")
    else:
        print(f"Modified MAPE: N/A (no non-zero values)")
    print(f"R² Score: {r2:.4f}")
    print("-" * 30)
    
    return {
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
        'mape': mape
    }

# Train and evaluate models for all datasets
# 1. Apartment 1 Kitchen
model_app1_kitchen, metrics_app1_kitchen = train_test_random_forest(
    X_train_app1_kitchen, y_train_app1_kitchen, 
    X_test_app1_kitchen, y_test_app1_kitchen,
    features_app1_kitchen, "Apartment 1 Kitchen"
)

# 2. Apartment 1 Office
model_app1_office, metrics_app1_office = train_test_random_forest(
    X_train_app1_office, y_train_app1_office, 
    X_test_app1_office, y_test_app1_office,
    features_app1_office, "Apartment 1 Office"
)

# 3. Apartment 2 Living Room
model_app2_living, metrics_app2_living = train_test_random_forest(
    X_train_app2_living, y_train_app2_living, 
    X_test_app2_living, y_test_app2_living,
    features_app2_living, "Apartment 2 Living Room"
)

# 4. Apartment 2 Office
model_app2_office, metrics_app2_office = train_test_random_forest(
    X_train_app2_office, y_train_app2_office, 
    X_test_app2_office, y_test_app2_office,
    features_app2_office, "Apartment 2 Office"
)

In [ ]:
os.makedirs('models', exist_ok=True)

# Save Apartment 1 Kitchen model
kitchen1_filename = "models/rf_apartment1_kitchen.pkl"
with open(kitchen1_filename, 'wb') as f:
    pickle.dump(model_app1_kitchen, f)

# Save Apartment 1 Office model
office1_filename = "models/rf_apartment1_office.pkl"
with open(office1_filename, 'wb') as f:
    pickle.dump(model_app1_office, f)

# Save Apartment 2 Living Room model
living2_filename = "models/rf_apartment2_living.pkl"
with open(living2_filename, 'wb') as f:
    pickle.dump(model_app2_living, f)

# Save Apartment 2 Office model
office2_filename = "models/rf_apartment2_office.pkl"
with open(office2_filename, 'wb') as f:
    pickle.dump(model_app2_office, f)

## Train-Test: XGBoost

In [ ]:
def train_test_xgboost(X_train, y_train, X_test, y_test, features, dataset_name):
    """
    Train an XGBoost regressor on the given dataset and evaluate performance.
    """
    print(f"\n{'='*80}")
    print(f"Training XGBoost Model for {dataset_name}")
    print(f"{'='*80}")
    
    # Create DMatrix objects for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)
    
    # Set parameters with stronger regularization for time series
    params = {
        'objective': 'reg:squarederror',
        'learning_rate': 0.01,          # Slower learning rate
        'max_depth': 6,                 # Reduced tree depth
        'min_child_weight': 5,          # Require more data in leaf nodes
        'subsample': 0.8,               # Use 80% of data per tree
        'colsample_bytree': 0.7,        # Use 70% of features per tree
        'gamma': 0.2,                   # Increased min loss reduction for split
        'alpha': 0.5,                   # Increased L1 regularization
        'lambda': 2.0,                  # Increased L2 regularization
        'seed': 42
    }
    
    # Train the model with early stopping
    eval_list = [(dtrain, 'train'), (dtest, 'test')]
    num_rounds = 400  # Increase max rounds (we lowered the learning rate)
    
    print("Training the model...")
    bst = xgb.train(
        params,
        dtrain,
        num_rounds,
        evals=eval_list,
        early_stopping_rounds=30,  # Increased early stopping patience
        verbose_eval=False
    )
    
    # Store the trained booster directly
    best_iteration = bst.best_iteration
    print(f"Best iteration: {best_iteration}")
    
    # Make predictions on training and test sets
    y_train_pred = bst.predict(dtrain)
    y_test_pred = bst.predict(dtest)
    
    # Evaluate the model
    print("\nModel Evaluation:")
    print("-" * 50)
    
    # Calculate metrics for both sets
    train_metrics = calculate_metrics(y_train, y_train_pred, "Training")
    test_metrics = calculate_metrics(y_test, y_test_pred, "Test")
    
    # Feature importance
    print("\nFeature Importance:")
    print("-" * 50)
    
    # Get feature importances
    importance_type = 'gain'  # 'gain' for the contribution to the model, 'weight' for number of splits
    feature_importances = bst.get_score(importance_type=importance_type)
    
    # Convert to dataframe for easier handling
    importance_df = pd.DataFrame({
        'Feature': list(feature_importances.keys()),
        'Importance': list(feature_importances.values())
    }).sort_values('Importance', ascending=False)
    
    # If not all features appear (XGBoost only includes used features),
    # add missing features with zero importance
    missing_features = set(features) - set(importance_df['Feature'])
    for feature in missing_features:
        importance_df = pd.concat([importance_df, pd.DataFrame({'Feature': [feature], 'Importance': [0]})], ignore_index=True)
    
    # Sort again after adding missing features
    importance_df = importance_df.sort_values('Importance', ascending=False)
    
    # Display top features
    print("Top 10 features by importance:")
    print(importance_df.head(10))
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    top_features = importance_df.head(10)
    plt.barh(np.arange(len(top_features)), top_features['Importance'], align='center')
    plt.yticks(np.arange(len(top_features)), top_features['Feature'])
    plt.xlabel('Importance')
    plt.title(f'Top 10 Important Features - {dataset_name}')
    plt.tight_layout()
    plt.show()
    
    # Return model and metrics
    metrics = {
        'train': train_metrics,
        'test': test_metrics,
        'feature_importance': importance_df,
        'best_iteration': best_iteration
    }
    
    return bst, metrics

def calculate_metrics(y_true, y_pred, set_name):
    """
    Calculate regression performance metrics
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # Alternative MAPE calculation that excludes zeros and is capped at 100%
    non_zero_mask = np.abs(y_true) > 0.1  # Only include values > 0.1
    if np.sum(non_zero_mask) > 0:
        mape_values = np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask]) * 100
        mape_capped = np.minimum(mape_values, 100)  # Cap at 100%
        mape = np.mean(mape_capped)
    else:
        mape = np.nan  # Not applicable if no non-zero values
    
    print(f"{set_name} Set Metrics:")
    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    if not np.isnan(mape):
        print(f"Modified MAPE (non-zero values only, capped): {mape:.2f}%")
    else:
        print(f"Modified MAPE: N/A (no non-zero values)")
    print(f"R² Score: {r2:.4f}")
    print("-" * 30)
    
    return {
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
        'mape': mape
    }

# Train and evaluate models for all datasets
# 1. Apartment 1 Kitchen
model_xgb_app1_kitchen, metrics_xgb_app1_kitchen = train_test_xgboost(
    X_train_app1_kitchen, y_train_app1_kitchen, 
    X_test_app1_kitchen, y_test_app1_kitchen,
    features_app1_kitchen, "Apartment 1 Kitchen"
)

# 2. Apartment 1 Office
model_xgb_app1_office, metrics_xgb_app1_office = train_test_xgboost(
    X_train_app1_office, y_train_app1_office, 
    X_test_app1_office, y_test_app1_office,
    features_app1_office, "Apartment 1 Office"
)

# 3. Apartment 2 Living Room
model_xgb_app2_living, metrics_xgb_app2_living = train_test_xgboost(
    X_train_app2_living, y_train_app2_living, 
    X_test_app2_living, y_test_app2_living,
    features_app2_living, "Apartment 2 Living Room"
)

# 4. Apartment 2 Office
model_xgb_app2_office, metrics_xgb_app2_office = train_test_xgboost(
    X_train_app2_office, y_train_app2_office, 
    X_test_app2_office, y_test_app2_office,
    features_app2_office, "Apartment 2 Office"
)

In [ ]:
os.makedirs('models', exist_ok=True)

# Save Apartment 1 Kitchen XGBoost model
kitchen1_filename = "models/xgb_apartment1_kitchen.pkl"
with open(kitchen1_filename, 'wb') as f:
    pickle.dump(model_xgb_app1_kitchen, f)

# Save Apartment 1 Office XGBoost model
office1_filename = "models/xgb_apartment1_office.pkl"
with open(office1_filename, 'wb') as f:
    pickle.dump(model_xgb_app1_office, f)

# Save Apartment 2 Living Room XGBoost model
living2_filename = "models/xgb_apartment2_living.pkl"
with open(living2_filename, 'wb') as f:
    pickle.dump(model_xgb_app2_living, f)

# Save Apartment 2 Office XGBoost model
office2_filename = "models/xgb_apartment2_office.pkl"
with open(office2_filename, 'wb') as f:
    pickle.dump(model_xgb_app2_office, f)

## Train-Test: Sarima

In [ ]:
def train_test_sarima(train_df, test_df, target_col, dataset_name, order=(1,1,1), seasonal_order=(1,1,1,24)):
    """
    Train a SARIMA model on time series data and evaluate performance.
    """
    print(f"\n{'='*80}")
    print(f"Training SARIMA Model for {dataset_name}")
    print(f"{'='*80}")
    
    # Ensure DataFrames have datetime indices
    if 'DateTime' in train_df.columns:
        train_df = train_df.set_index('DateTime')
    if 'DateTime' in test_df.columns:
        test_df = test_df.set_index('DateTime')
    
    # Extract target series
    y_train = train_df[target_col]
    y_test = test_df[target_col]
    
    print(f"Training data shape: {y_train.shape}")
    print(f"Test data shape: {y_test.shape}")
    print(f"SARIMA Order: {order}, Seasonal Order: {seasonal_order}")
    
    # Train SARIMA model
    print("Training SARIMA model...")
    model = SARIMAX(
        y_train,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    
    results = model.fit(disp=False)  # Suppress convergence details
    print("Model fitting complete.")
    
    # Generate predictions
    print("Generating predictions...")
    y_train_pred = results.predict(start=0, end=len(y_train)-1)
    
    # Get forecast for test period
    y_test_pred = results.get_forecast(steps=len(y_test)).predicted_mean
    
    # If index doesn't match, align by position
    if len(y_test) != len(y_test_pred) or not all(y_test.index == y_test_pred.index):
        print("Warning: Test predictions index doesn't match. Aligning by position.")
        y_test_pred = pd.Series(y_test_pred.values[:len(y_test)], index=y_test.index)
    
    # Evaluate model
    print("\nModel Evaluation:")
    print("-" * 50)
    
    # Calculate metrics for both sets
    train_metrics = calculate_metrics(y_train, y_train_pred, "Training")
    test_metrics = calculate_metrics(y_test, y_test_pred, "Test")
    
    # Summary of SARIMA model
    print("\nModel Summary:")
    print("-" * 50)
    print(results.summary().tables[0].as_text())
    
    # Return results
    metrics = {
        'train': train_metrics,
        'test': test_metrics,
        'aic': results.aic,
        'bic': results.bic
    }
    
    return results, metrics

def calculate_metrics(y_true, y_pred, set_name):
    """
    Calculate regression performance metrics
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # Alternative MAPE calculation that excludes zeros and is capped at 100%
    non_zero_mask = np.abs(y_true) > 0.1  # Only include values > 0.1
    if np.sum(non_zero_mask) > 0:
        mape_values = np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask]) * 100
        mape_capped = np.minimum(mape_values, 100)  # Cap at 100%
        mape = np.mean(mape_capped)
    else:
        mape = np.nan  # Not applicable if no non-zero values
    
    print(f"{set_name} Set Metrics:")
    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    if not np.isnan(mape):
        print(f"Modified MAPE (non-zero values only, capped): {mape:.2f}%")
    else:
        print(f"Modified MAPE: N/A (no non-zero values)")
    print(f"R² Score: {r2:.4f}")
    print("-" * 30)
    
    return {
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
        'mape': mape
    }

# SARIMA parameters for each room - tuned for hourly data with daily seasonality
sarima_params = {
    'Apartment_1_Kitchen': ((1,1,1), (1,1,1,24)),
    'Apartment_1_Office': ((1,1,1), (0,1,1,24)),
    'Apartment_2_Living': ((2,1,1), (1,1,1,24)),
    'Apartment_2_Office': ((1,1,2), (1,1,1,24))
}

# Train and evaluate SARIMA models for all datasets
# 1. Apartment 1 Kitchen
model_sarima_app1_kitchen, metrics_sarima_app1_kitchen = train_test_sarima(
    train_df_app1_kitchen, 
    test_df_app1_kitchen, 
    target_app1_kitchen,
    "Apartment 1 Kitchen",
    order=sarima_params['Apartment_1_Kitchen'][0],
    seasonal_order=sarima_params['Apartment_1_Kitchen'][1]
)

# 2. Apartment 1 Office
model_sarima_app1_office, metrics_sarima_app1_office = train_test_sarima(
    train_df_app1_office, 
    test_df_app1_office, 
    target_app1_office,
    "Apartment 1 Office",
    order=sarima_params['Apartment_1_Office'][0],
    seasonal_order=sarima_params['Apartment_1_Office'][1]
)

# 3. Apartment 2 Living Room
model_sarima_app2_living, metrics_sarima_app2_living = train_test_sarima(
    train_df_app2_living, 
    test_df_app2_living, 
    target_app2_living,
    "Apartment 2 Living Room",
    order=sarima_params['Apartment_2_Living'][0],
    seasonal_order=sarima_params['Apartment_2_Living'][1]
)

# 4. Apartment 2 Office
model_sarima_app2_office, metrics_sarima_app2_office = train_test_sarima(
    train_df_app2_office, 
    test_df_app2_office, 
    target_app2_office,
    "Apartment 2 Office",
    order=sarima_params['Apartment_2_Office'][0],
    seasonal_order=sarima_params['Apartment_2_Office'][1]
)


In [ ]:
os.makedirs('models', exist_ok=True)

# Save Apartment 1 Kitchen SARIMA model
kitchen1_filename = "models/sarima_apartment1_kitchen.pkl"
with open(kitchen1_filename, 'wb') as f:
    pickle.dump(model_sarima_app1_kitchen, f)

# Save Apartment 1 Office SARIMA model
office1_filename = "models/sarima_apartment1_office.pkl"
with open(office1_filename, 'wb') as f:
    pickle.dump(model_sarima_app1_office, f)

# Save Apartment 2 Living Room SARIMA model
living2_filename = "models/sarima_apartment2_living.pkl"
with open(living2_filename, 'wb') as f:
    pickle.dump(model_sarima_app2_living, f)

# Save Apartment 2 Office SARIMA model
office2_filename = "models/sarima_apartment2_office.pkl"
with open(office2_filename, 'wb') as f:
    pickle.dump(model_sarima_app2_office, f)